# Практика: фильтры и типы

Тот же `listings.csv`. Фильтры — первый инструмент аналитика: «покажи только центр», «только вместительные объявления».

In [ ]:
from pathlib import Path
import pandas as pd


def find_listings_csv() -> Path:
    for p in (Path('listings.csv'), Path('../../data/listings.csv'), Path('../data/listings.csv')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError('listings.csv не найден')


LISTINGS_PATH = find_listings_csv()
df = pd.read_csv(LISTINGS_PATH)


## 1. Фильтр по району

Оставьте строки с `neighbourhood == 'center'`. Сколько таких объявлений?

**Why:** маска — Series из True/False той же длины, что `df`.

**Вопрос:** строка в `center` — подмножество таблицы или новый файл?

In [ ]:
center = None  # df[маска]
assert center is not None
assert (center['neighbourhood'] == 'center').all()
assert len(center) > 0
print('center:', len(center), 'из', len(df))

## 2. Вместительные объявления

`accommodates >= 6`. Выведите `listing_id`, вместимость, цена (`head`).

**Pitfall:** сравнивайте с числом (`6`), не со строкой `'6'`.

In [ ]:
large_listings = None
assert large_listings is not None
assert (large_listings['accommodates'] >= 6).all()
print(large_listings[['listing_id', 'accommodates', 'price']].head())

## 3. Составная маска (&)

Нужны объявления **и** в center, **и** с `accommodates >= 4`. Так **не работает**: `df['neighbourhood'] == 'center' and df['accommodates'] >= 4` — Python сравнивает два Series через `and` и падает.

**How:** `&` и скобки вокруг каждого условия: `df[(...) & (...)]`.

In [ ]:
center_long = None  # df[(...) & (...)]
assert center_long is not None
assert (center_long['neighbourhood'] == 'center').all()
assert (center_long['accommodates'] >= 4).all()
print(len(center_long))

## 4. Отрицание ~

Оставьте объявления **не** из `center` (`neighbourhood != 'center'` или `~ (df['neighbourhood'] == 'center')`).

**Checkpoint:** `len(not_center) + len(center)` должно равняться `len(df)` (если нет NaN в neighbourhood).

In [ ]:
not_center = None
assert not_center is not None
assert (not_center['neighbourhood'] != 'center').all()
assert len(not_center) + len(center) == len(df)
print('not center:', len(not_center))

## 5. Сортировка и топ-N

Три самые дорогие объявления по `price` — только `listing_id` и `price`.

**How:** `sort_values(..., ascending=False).head(3)`.

In [ ]:
top3 = None
assert top3 is not None
assert len(top3) == 3
assert top3['price'].is_monotonic_decreasing
print(top3[['listing_id', 'price']])

## 6. Доля фильтра

Доля объявлений `neighbourhood == 'center'` среди всех (0–1, округлить до 3 знаков).

**Why:** абсолютное число строк без доли плохо сравнивать между выборками разного размера.

In [ ]:
share_center = None
assert share_center is not None
assert 0 < float(share_center) < 1
print('доля center:', share_center)

## 7. loc vs iloc после фильтра

После фильтра `center` индекс может быть «дырявым». Возьмите **первую строку** `center` через `.iloc[0]` и запишите её `listing_id`.

**Pitfall:** `center.loc[0]` часто падает — метки 0 может не быть.

In [ ]:
first_center_id = None  # str listing_id
assert first_center_id is not None
assert isinstance(first_center_id, str) and first_center_id.startswith('L')
print(first_center_id)

## 8. Тип number_of_reviews

Приведите `number_of_reviews` к `int`, проверьте, что значения ≥ 0. Число отзывов — **счётчик**, не доля и не цена.

In [ ]:
reviews = None  # df['number_of_reviews'].astype(int)
assert reviews is not None
assert reviews.dtype.kind in 'iu'
assert reviews.min() >= 0
print(int(reviews.min()), int(reviews.max()))